In [1]:
import os
import yfinance as yf
import pandas as pd
import pandas_ta as ta
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# 1. Pobranie i przygotowanie danych
symbol = os.environ.get("STOCK_TICKER", "").strip().upper()
if not symbol:
    raise RuntimeError("Set STOCK_TICKER before running price_extractor.ipynb")
df = yf.Ticker(symbol).history(period="10y")
df.index = df.index.tz_localize(None)

# Wskaźniki i sygnały
df.ta.macd(fast=12, slow=26, signal=9, append=True)
df.ta.rsi(length=9, append=True)
df.ta.rsi(length=14, append=True)
df.ta.rsi(length=21, append=True)
df.dropna(inplace=True)

macd = df['MACD_12_26_9']
signal = df['MACDs_12_26_9']
wczorajszy_macd = macd.shift(1)
wczorajszy_signal = signal.shift(1)

df['Decyzja'] = 'BRAK'
df.loc[(macd > signal) & (wczorajszy_macd <= wczorajszy_signal), 'Decyzja'] = 'KUPUJ'
df.loc[(macd < signal) & (wczorajszy_macd >= wczorajszy_signal), 'Decyzja'] = 'SPRZEDAJ'

df_kupuj = df[df['Decyzja'] == 'KUPUJ']
df_sprzedaj = df[df['Decyzja'] == 'SPRZEDAJ']
df_dywidendy = df[df['Dividends'] > 0]
df_splity = df[df['Stock Splits'] > 0]

# Wspólna konfiguracja osi X
konfiguracja_osi_x = dict(
    type="date",
    showspikes=True, spikemode="across", spikesnap="cursor", spikedash="dot", spikecolor="#ffffff", spikethickness=1, hoverformat="%Y-%m-%d",
    rangeselector=dict(
        buttons=list([
            dict(count=1, label="1M", step="month", stepmode="backward"),
            dict(count=3, label="3M", step="month", stepmode="backward"),
            dict(count=6, label="6M", step="month", stepmode="backward"),
            dict(count=1, label="YTD", step="year", stepmode="todate"),
            dict(count=1, label="1Y", step="year", stepmode="backward"),
            dict(count=5, label="5Y", step="year", stepmode="backward"),
            dict(step="all", label="10Y")
        ]),
        bgcolor="#333333", activecolor="#2962ff", x=0, y=1.1
    )
)

# ==========================================
# WYKRES 1: CENA + WOLUMEN + SYGNAŁY
# ==========================================
fig1 = make_subplots(specs=[[{"secondary_y": True}]])

kolory_wolumenu = ['#26a69a' if row['Close'] >= row['Open'] else '#ef5350' for index, row in df.iterrows()]
fig1.add_trace(go.Bar(x=df.index, y=df['Volume'], marker_color=kolory_wolumenu, opacity=0.25, name='Wolumen'), secondary_y=True)
fig1.add_trace(go.Candlestick(x=df.index, open=df['Open'], high=df['High'], low=df['Low'], close=df['Close'], name='Cena', increasing_line_color='#26a69a', decreasing_line_color='#ef5350'), secondary_y=False)
fig1.add_trace(go.Scatter(x=df_kupuj.index, y=df_kupuj['Low'] * 0.95, customdata=df_kupuj[['Low', 'High']], mode='markers', marker=dict(symbol='triangle-up', size=16, color='#00ff00', line=dict(width=1, color='black')), name='KUPUJ', hovertemplate='KUPUJ<br>Najniższa: %{customdata[0]:.2f}<br>Najwyższa: %{customdata[1]:.2f}<extra></extra>'), secondary_y=False)
fig1.add_trace(go.Scatter(x=df_sprzedaj.index, y=df_sprzedaj['High'] * 1.05, customdata=df_sprzedaj[['Low', 'High']], mode='markers', marker=dict(symbol='triangle-down', size=16, color='#ff0000', line=dict(width=1, color='black')), name='SPRZEDAJ', hovertemplate='SPRZEDAJ<br>Najniższa: %{customdata[0]:.2f}<br>Najwyższa: %{customdata[1]:.2f}<extra></extra>'), secondary_y=False)

if not df_dywidendy.empty:
    fig1.add_trace(go.Scatter(x=df_dywidendy.index, y=df_dywidendy['Low'] * 0.90, mode='markers+text', marker=dict(symbol='circle', size=12, color='#29b6f6', line=dict(width=1, color='white')), text=df_dywidendy['Dividends'].apply(lambda x: f"{x}$"), textposition="bottom center", name='Dywidenda', showlegend=False, hovertemplate='Dywidenda: %{text}<extra></extra>', textfont=dict(color="#29b6f6")), secondary_y=False)
if not df_splity.empty:
                    fig1.add_trace(go.Scatter(x=df_splity.index, y=df_splity['High'], mode='markers+text', marker=dict(symbol='star', size=16, color='#ffee58', line=dict(width=1, color='black')), text=df_splity['Stock Splits'].apply(lambda x: f"{x:g}:1" if x >= 1 else f"1:{1 / x:g}"), textposition="top center", name='Split', hovertemplate='%{text}<extra></extra>', textfont=dict(color="#ffee58")), secondary_y=False)

fig1.update_layout(title=dict(text=f"Cena Akcji, Wolumen i Sygnały Transakcyjne ({symbol})", x=0.5, xanchor='center'), dragmode='pan', xaxis_rangeslider_visible=False, template='plotly_dark', height=600, hovermode='x unified', margin=dict(l=60, r=40, t=60, b=40))
fig1.update_xaxes(**konfiguracja_osi_x)
fig1.update_xaxes(range=[df.index.max() - pd.Timedelta(days=180), df.index.max()])
fig1.update_yaxes(title_text="Cena (USD)", secondary_y=False, gridcolor='#333333')
fig1.update_yaxes(range=[0, df['Volume'].max() * 4], showgrid=False, showticklabels=False, secondary_y=True)
fig1.show(config={'scrollZoom': True})

# ==========================================
# WYKRES 2: WSKAŹNIK MACD
# ==========================================
fig2 = go.Figure()
fig2.add_trace(go.Scatter(x=df.index, y=df['MACD_12_26_9'], line=dict(color='#2962ff', width=2.5), name='MACD'))
fig2.add_trace(go.Scatter(x=df.index, y=df['MACDs_12_26_9'], line=dict(color='#ff6d00', width=2), name='Sygnał'))
kolory_macd = ['#26a69a' if val >= 0 else '#ef5350' for val in df['MACDh_12_26_9']]
fig2.add_trace(go.Bar(x=df.index, y=df['MACDh_12_26_9'], marker_color=kolory_macd, name='Histogram', opacity=0.7))
fig2.update_layout(title=dict(text="Wskaźnik Trendu: MACD (12, 26, 9)", x=0.5, xanchor='center'), dragmode='pan', xaxis_rangeslider_visible=False, template='plotly_dark', height=400, hovermode='x unified', margin=dict(l=60, r=40, t=60, b=40))
fig2.update_xaxes(**konfiguracja_osi_x)
fig2.update_xaxes(range=[df.index.max() - pd.Timedelta(days=180), df.index.max()])
fig2.update_yaxes(title_text="MACD", gridcolor='#333333')
fig2.show(config={'scrollZoom': True})

# ==========================================
# WYKRES 3: OSCYLATOR RSI (9)
# ==========================================
fig3 = go.Figure()
fig3.add_trace(go.Scatter(x=df.index, y=df['RSI_9'], line=dict(color='#00e5ff', width=2), name='RSI (9)'))
fig3.add_hline(y=70, line_dash="dash", line_color="#ef5350", line_width=1.5)
fig3.add_hline(y=30, line_dash="dash", line_color="#26a69a", line_width=1.5)
fig3.add_hrect(y0=30, y1=70, line_width=0, fillcolor="purple", opacity=0.1)
fig3.update_layout(title=dict(text="Oscylator Siły Względnej: RSI (9)", x=0.5, xanchor='center'), dragmode='pan', xaxis_rangeslider_visible=False, template='plotly_dark', height=400, hovermode='x unified', margin=dict(l=60, r=40, t=60, b=40))
fig3.update_xaxes(**konfiguracja_osi_x)
fig3.update_xaxes(range=[df.index.max() - pd.Timedelta(days=180), df.index.max()])
fig3.update_yaxes(title_text="RSI (9)", range=[0, 100], gridcolor='#333333')
fig3.show(config={'scrollZoom': True})

# ==========================================
# WYKRES 4: OSCYLATOR RSI (14)
# ==========================================
fig4 = go.Figure()
fig4.add_trace(go.Scatter(x=df.index, y=df['RSI_14'], line=dict(color='#d500f9', width=2), name='RSI (14)'))
fig4.add_hline(y=70, line_dash="dash", line_color="#ef5350", line_width=1.5)
fig4.add_hline(y=30, line_dash="dash", line_color="#26a69a", line_width=1.5)
fig4.add_hrect(y0=30, y1=70, line_width=0, fillcolor="purple", opacity=0.1)
fig4.update_layout(title=dict(text="Oscylator Siły Względnej: RSI (14)", x=0.5, xanchor='center'), dragmode='pan', xaxis_rangeslider_visible=False, template='plotly_dark', height=400, hovermode='x unified', margin=dict(l=60, r=40, t=60, b=40))
fig4.update_xaxes(**konfiguracja_osi_x)
fig4.update_xaxes(range=[df.index.max() - pd.Timedelta(days=180), df.index.max()])
fig4.update_yaxes(title_text="RSI (14)", range=[0, 100], gridcolor='#333333')
fig4.show(config={'scrollZoom': True})

# ==========================================
# WYKRES 5: OSCYLATOR RSI (21)
# ==========================================
fig5 = go.Figure()
fig5.add_trace(go.Scatter(x=df.index, y=df['RSI_21'], line=dict(color='#64dd17', width=2), name='RSI (21)'))
fig5.add_hline(y=70, line_dash="dash", line_color="#ef5350", line_width=1.5)
fig5.add_hline(y=30, line_dash="dash", line_color="#26a69a", line_width=1.5)
fig5.add_hrect(y0=30, y1=70, line_width=0, fillcolor="purple", opacity=0.1)
fig5.update_layout(title=dict(text="Oscylator Siły Względnej: RSI (21)", x=0.5, xanchor='center'), dragmode='pan', xaxis_rangeslider_visible=False, template='plotly_dark', height=400, hovermode='x unified', margin=dict(l=60, r=40, t=60, b=40))
fig5.update_xaxes(**konfiguracja_osi_x)
fig5.update_xaxes(range=[df.index.max() - pd.Timedelta(days=180), df.index.max()])
fig5.update_yaxes(title_text="RSI (21)", range=[0, 100], gridcolor='#333333')
fig5.show(config={'scrollZoom': True})